# Calculating Pixel IVT Limits
This notebook will calculate the pixel IVT limits for tARget. Unless otherwise stated, everything that has to do with tARget is coded in MATLAB to avoid data structure conflicts.

In [1]:
addpath('..')
wusd3 = readtable('wusd3_gcms.csv');
wusd3.(4)(7) = {'nan'};

for i = 1011:20:1201
    le_row = {16+(i-1011)/20 , 'cesm2-le', 'CESM2-LE', num2str(i), '365_day'};
    wusd3 = [wusd3;le_row];
end
wusd3 % Load the wusd3_gcms table and add all the CESM2-LE members to it.

Set 'VariableNamingRule' to 'preserve' to use the original column headers as table variable names.

wusd3 =

  26x5 table

    Var1          Model              ModelName           Member          Calendar  
    ____    _________________    _________________    _____________    ____________

      0     {'access-cm2'   }    {'ACCESS-CM2'   }    {'r5i1p1f1' }    {'standard'}
      1     {'canesm5'      }    {'CanESM5'      }    {'r1i1p2f1' }    {'365_day' }
      2     {'cesm2'        }    {'CESM2'        }    {'r11i1p1f1'}    {'365_day' }
      3     {'cnrm-esm2-1'  }    {'CNRM-ESM2-1'  }    {'r1i1p1f2' }    {'standard'}
      4     {'ec-earth3'    }    {'EC-Earth3'    }    {'r1i1p1f1' }    {'standard'}
      5     {'ec-earth3-veg'}    {'EC-Earth3-Veg'}    {'r1i1p1f1' }    {'standard'}
      6     {'era5'         }    {'ERA5'         }    {'nan'      }    {'standard'}
      7     {'fgoals-g3'    }    {'FGOALS-g3'    }    {'r1i1p1f1' }    {'365_day' }
      8     {'giss-e2-1-g'  }    {'G

In [2]:
for idx=1:height(wusd3) % Run through each IVT model
    model = wusd3.Model{idx};
    MName = wusd3.ModelName{idx};
    mem = wusd3.Member{idx};
    cal = wusd3.Calendar{idx};
    
    % define ivt file
    ivtfilec = sprintf('/glade/derecho/scratch/tcorrie/regrids/ivt.daily.%s.%s.d01_regridded_tARgetivtcalc.nc', model, mem);
    sdate = ncreadatt(ivtfilec, 'time', 'units');
    cal = ncreadatt(ivtfilec, 'time', 'calendar');
    % Build timedef array
    nsteps = ncinfo(ivtfilec, 'time').Size;
    year = str2num(sdate(12:15));
    month = str2num(sdate(17:18));
    day = str2num(sdate(20:21));
    hour = 0;
    dt = 24;
    % Pass other files through here
    
    ivtfile = ivtfilec; % Argument1: Historical file 
 
    islndfile = []; % not using this
    ivtxname = 'ivtx';
    ivtyname = 'ivty';
    pixelarea = []; % N/A b/c re-gridding to regular lat/lon grid
    if strcmp(cal, 'noleap')
        timedef = [nsteps, year, month, day, hour, dt, 365];
    elseif strcmp(cal, '360_day')
        timedef = [nsteps, year, month, day, hour, dt, 360];
    else
        timedef = [nsteps, year, month, day, hour, dt];
    end 
    
    % Define outfile but really only for path to putting pixel_ivt_limit.nc where it needs to go.
    outfile = '/glade/derecho/scratch/tcorrie/tARget/pixel_ivt_limits/ofile.nc';
    
    % Other limit stuff
    pixel_ivt_percentile_limits = [85, 87.5, 90, 92.5, 95];
    global_ivt_percentile_limit = 5; % Doesn't change
    universal_ivt_limits = [0,250]; % Anything over 250 always kept
    length_limit = 2e6; % Doesn't change. 
    lenwidratio_limit = 2; % Doesn't change.
    ivtonly = true; % Only getting IVT limits here
    % The actual running of tARget but only to get the specific pixel IVT limit.
    pixel_ivt_filename = strcat('pixel_ivt_limit_', model, '_', mem, '.nc');

    % target_shapeonly was an attempt to try and remove the time stitching piece of tARget to speed up the process of
    %  running tARget on multiple GCMs at once. Results are inconclusive but this DOES work. One thing I've added (aside
    %  from ivtonly) is an option to set pixel_ivt_filename to something other than the default name. If not passed, it
    %  will use the default name.
    target_shapeonly(ivtfile,islndfile,ivtxname,ivtyname,pixelarea,timedef,-9999, ...
    outfile,pixel_ivt_percentile_limits,global_ivt_percentile_limit,universal_ivt_limits, ...
    length_limit,lenwidratio_limit,ivtonly,pixel_ivt_filename);
end

Tracking Atmospheric Rivers Globally as Elongated Targets (tARget), Version 4
Copyright (c) 2015-2024, Bin Guan. All rights reserved.
 
References:
- Guan, B., and D. E. Waliser (2015), Detection of atmospheric rivers:
  Evaluation and application of an algorithm for global studies,
  J. Geophys. Res. Atmos., 120, 12514-12535, doi:10.1002/2015JD024257.
- Guan, B., D. E. Waliser, and F. M. Ralph (2018), An inter-comparison
  between reanalysis and dropsonde observations of the total water vapor
  transport in individual atmospheric rivers, J. Hydrometeorol., 19,
  321-337, doi:10.1175/JHM-D-17-0114.1.
- Guan, B., and D. E. Waliser (2019), Tracking atmospheric rivers globally:
  Spatial distributions and temporal evolution of life cycle characteristics,
  J. Geophys. Res. Atmos., 124, 12523-12552, doi:10.1029/2019JD031205.
- Guan, B., and D. E. Waliser (2024), A regionally refined quarter-degree
  global atmospheric rivers database based on ERA5, Sci. Data, accepted.
 
Obtaining licenses

In [ ]:
%parpool(1000)

In [ ]:
%parfor idx=1:10
%    member = 1011+20*(idx-1)
%    feval(@targetlimits4ensemble, member);
%end